Load Data

In [ ]:
#load Thư Viện
# Import toàn bộ thư viện chính trong requirements.txt
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import cv2
import mediapipe as mp

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import joblib

print("Python:", sys.version)
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn OK")

In [ ]:
#Load 
BASE_DIR = Path(r"C:\Users\KIET PC\Desktop\Focus_Guard")

DATASET_DIR = BASE_DIR / "DataSet"
LABELS_DIR = BASE_DIR / "Labels"

print("BASE_DIR exists:", BASE_DIR.exists())
print("DataSet exists:", DATASET_DIR.exists())
print("Labels exists:", LABELS_DIR.exists())

In [ ]:
label_files = list(LABELS_DIR.glob("*.csv"))
print(f"Tìm thấy {len(label_files)} file CSV trong Labels/:")
for f in label_files:
    print(" -", f.name)

In [ ]:
# TODO: sửa lại tên file nếu khác so với mặc định bên dưới
train_labels = pd.read_csv(LABELS_DIR / "TrainLabels.csv")
val_labels   = pd.read_csv(LABELS_DIR / "ValidationLabels.csv")
test_labels  = pd.read_csv(LABELS_DIR / "TestLabels.csv")

print("Train:", train_labels.shape)
print("Val:  ", val_labels.shape)
print("Test: ", test_labels.shape)

train_labels.head()

In [ ]:
# Strip khoảng trắng thừa trong tên cột (lỗi phổ biến của DAiSEE gốc)
for df in [train_labels, val_labels, test_labels]:
    df.columns = df.columns.str.strip()

print(train_labels.columns.tolist())

In [ ]:
dimensions = ["Boredom", "Engagement", "Confusion", "Frustration"]
dimensions = [d for d in dimensions if d in train_labels.columns]

for dim in dimensions:
    print(f"--- {dim} ---")
    print(train_labels[dim].value_counts().sort_index())
    print()

#####

In [ ]:
"""
train_models.py
Train DT, RF, XGBoost tren train_features.csv, danh gia tren val/test_features.csv.
Moi dimension (Engagement/Boredom/Confusion/Frustration) train 1 model rieng.

Cach dung:
    python train_models.py
(sua BASE_DIR ben duoi cho dung may ban truoc khi chay)
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from xgboost import XGBClassifier


# ---------------------------------------------------------------------
# CAU HINH -- SUA CHO DUNG MAY BAN
# ---------------------------------------------------------------------
BASE_DIR = Path(r"C:\Users\KIET PC\Desktop\Focus_Guard")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "outputs" / "models"
RESULTS_DIR = BASE_DIR / "outputs" / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLUMNS = ["Boredom", "Engagement", "Confusion", "Frustration"]

# Cac cot feature -- 20 cot da nen tu extract_features.py (khong lay ClipID/label/QA columns)
FEATURE_PREFIXES = ["ear_", "mar_", "yaw_", "pitch_"]


def get_feature_columns(df: pd.DataFrame) -> list:
    return [c for c in df.columns if any(c.startswith(p) for p in FEATURE_PREFIXES)]


def load_data():
    train_df = pd.read_csv(PROCESSED_DIR / "train_features.csv")
    val_df = pd.read_csv(PROCESSED_DIR / "val_features.csv")
    test_df = pd.read_csv(PROCESSED_DIR / "test_features.csv")

    # Loai bo dong bi NaN (video khong detect duoc frame nao -> feature rong)
    feature_cols = get_feature_columns(train_df)

    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        before = len(df)
        df.dropna(subset=feature_cols, inplace=True)
        after = len(df)
        if before != after:
            print(f"  {name}: bo {before - after} dong bi NaN (khong detect duoc face)")

    return train_df, val_df, test_df, feature_cols


def train_one_label(model_name: str, model_class, model_kwargs: dict,
                     X_train, y_train, X_val, y_val, X_test, y_test, label_name: str):
    """Train + danh gia 1 model cho 1 label (vd Random Forest cho Boredom)."""
    model = model_class(**model_kwargs)
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    result = {
        "model": model_name,
        "label": label_name,
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_f1_macro": f1_score(y_val, val_pred, average="macro", zero_division=0),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_f1_macro": f1_score(y_test, test_pred, average="macro", zero_division=0),
        "test_confusion_matrix": confusion_matrix(y_test, test_pred).tolist(),
        "test_classification_report": classification_report(y_test, test_pred, zero_division=0, output_dict=True),
    }

    return model, result


def main():
    print("Loading data...")
    train_df, val_df, test_df, feature_cols = load_data()
    print(f"Feature columns ({len(feature_cols)}): {feature_cols}\n")

    X_train = train_df[feature_cols].values
    X_val = val_df[feature_cols].values
    X_test = test_df[feature_cols].values

    # 3 thuat toan can train, moi cai co hyperparameter rieng
    algorithms = {
        "DecisionTree": (
            DecisionTreeClassifier,
            {"max_depth": 8, "min_samples_leaf": 10, "class_weight": "balanced", "random_state": 42},
        ),
        "RandomForest": (
            RandomForestClassifier,
            {"n_estimators": 200, "max_depth": 12, "class_weight": "balanced", "random_state": 42, "n_jobs": -1},
        ),
        "XGBoost": (
            XGBClassifier,
            {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1,
             "eval_metric": "mlogloss", "random_state": 42},
        ),
    }

    all_results = []

    for model_name, (model_class, model_kwargs) in algorithms.items():
        print(f"\n{'='*60}\nTraining {model_name}\n{'='*60}")

        for label_name in LABEL_COLUMNS:
            y_train = train_df[label_name].values
            y_val = val_df[label_name].values
            y_test = test_df[label_name].values

            model, result = train_one_label(
                model_name, model_class, model_kwargs,
                X_train, y_train, X_val, y_val, X_test, y_test, label_name
            )

            print(f"  [{label_name}] val_acc={result['val_accuracy']:.3f}  "
                  f"val_f1={result['val_f1_macro']:.3f}  "
                  f"test_acc={result['test_accuracy']:.3f}  "
                  f"test_f1={result['test_f1_macro']:.3f}")

            # Luu model
            model_path = MODELS_DIR / f"{model_name}_{label_name}.pkl"
            joblib.dump(model, model_path)

            all_results.append(result)

    # Luu toan bo ket qua ra 1 file json duy nhat -- dung cho notebook so sanh sau nay
    results_path = RESULTS_DIR / "all_results.json"
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print(f"\n\nDa luu toan bo model vao: {MODELS_DIR}")
    print(f"Da luu ket qua vao: {results_path}")

    # In bang tong ket ngan gon
    print(f"\n{'='*60}\nTONG KET (test_f1_macro)\n{'='*60}")
    summary_df = pd.DataFrame(all_results)[["model", "label", "test_f1_macro", "test_accuracy"]]
    pivot = summary_df.pivot(index="label", columns="model", values="test_f1_macro")
    print(pivot.round(3))


if __name__ == "__main__":
    main()